# Bài 4: Phát hiện Tấn công ICS bằng Mô hình CNN (1D Convolutional Neural Network)

## Mục tiêu
- Xây dựng 1D CNN cho phân loại chuỗi thời gian ICS
- Hiểu Conv1D, BatchNorm, MaxPooling, GlobalAveragePooling
- So sánh CNN với LSTM về hiệu năng và tốc độ

## Hướng dẫn
Điền code vào các ô `???`. Tham khảo `b4_cnn_demo.ipynb` nếu cần.

## Phần 1: Import thư viện

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              confusion_matrix, roc_auc_score, roc_curve,
                              ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, BatchNormalization,
                                      GlobalAveragePooling1D, Dense, Dropout, Input)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)
plt.rcParams['figure.figsize'] = (13, 5)
print(f'TensorFlow version: {tf.__version__}')
print('Thư viện đã được import thành công!')

## Phần 2: Tải và Chuẩn bị Dữ liệu

In [ ]:
train_df = pd.read_csv('data/BATADAL/train_dataset.csv', index_col=0)
test1_df = pd.read_csv('data/BATADAL/test_dataset_1.csv', index_col=0)
test2_df = pd.read_csv('data/BATADAL/test_dataset_2.csv', index_col=0)

feature_cols = [c for c in train_df.columns if c not in ['DATETIME', 'ATT_FLAG']]
n_features   = len(feature_cols)

# Fit scaler tren train_df (du lieu binh thuong thuan tuy) — dung hon fit tren test1
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(train_df[feature_cols].values)
X_test1_sc = scaler.transform(test1_df[feature_cols].values)
X_test2_sc = scaler.transform(test2_df[feature_cols].values)
y_test1    = test1_df['ATT_FLAG'].values
y_test2    = test2_df['ATT_FLAG'].values

print(f'Train : {train_df.shape[0]} mau binh thuong (dung de fit scaler)')
print(f'Test 1: {test1_df.shape[0]} mau  (Normal={(y_test1==0).sum()}, Attack={(y_test1==1).sum()})')
print(f'Test 2: {test2_df.shape[0]} mau  (Normal={(y_test2==0).sum()}, Attack={(y_test2==1).sum()})')
print(f'So dac trung: {n_features}')

## Phần 3: Tạo Chuỗi bằng Cửa sổ Trượt (Temporal Split + Augment)

**Vấn đề data leakage với `train_test_split` ngẫu nhiên:**

Khi dùng `STEP=1` rồi `train_test_split` ngẫu nhiên, sequence `[0:30]` trong train
và sequence `[1:31]` trong val chồng lấp 29/30 timestep → model ghi nhớ test1 hoàn hảo
nhưng không tổng quát hóa được sang test2 (AUC chỉ ~0.65).

**Giải pháp: Temporal Split + Data Augmentation**
1. Chia raw time series test1 theo **thứ tự thời gian** (70% đầu = train, 30% cuối = val)
2. Tạo sequences từ mỗi phần riêng biệt — không có overlap qua ranh giới
3. **Augment** tập train bằng normal sequences từ `train_df` (48K mẫu bình thường thuần túy)
   → Mô hình học được dạng bình thường đa dạng hơn, cải thiện khả năng tổng quát hóa
4. Dùng `STEP=5` cho train (giảm chồng lấp), `STEP=8` cho normal augmentation, `STEP=1` cho val/test

In [ ]:
WINDOW_SIZE  = 30
STEP_TRAIN   = 5   # stride khi tao train sequences (giam chong lap)
STEP_NORM    = 8   # stride cho normal sequences tu train_df
STEP_EVAL    = 1   # stride cho val va test

def create_sequences(X, y, window_size=30, step=1):
    sequences, labels = [], []
    for i in range(0, len(X) - window_size + 1, step):
        # TODO: lay chuoi tu vi tri i den i+window_size
        sequences.append(X[i : i + ???])
        labels.append(1 if y[i : i + window_size].sum() > 0 else 0)
    return np.array(sequences, dtype=np.float32), np.array(labels, dtype=np.int32)

# Temporal split: chia raw series TEST truoc, tao sequences sau (tranh data leakage)
split_idx = int(len(X_test1_sc) * ???)   # 70% dau lam train

# --- Tap train: ket hop normal tu train_df + attack tu test1 (70% dau) ---
# TODO: Dien nhan 0 cho toan bo mau binh thuong tu train_df
y_train_zeros = np.zeros(len(X_train_sc), dtype=np.int32)
X_norm_seq, y_norm_seq = create_sequences(???, y_train_zeros,
                                           WINDOW_SIZE, step=STEP_NORM)

# TODO: Dien phan dau test1 (truoc split_idx) de tao sequences train
X_t1_tr, y_t1_tr = create_sequences(???,
                                     y_test1[:split_idx], WINDOW_SIZE, STEP_TRAIN)

# Ket hop va tron ngau nhien
X_tr = np.concatenate([X_norm_seq, X_t1_tr], axis=0)
y_tr = np.concatenate([y_norm_seq, y_t1_tr], axis=0)
perm = np.random.default_rng(42).permutation(len(X_tr))
X_tr, y_tr = X_tr[perm], y_tr[perm]

# --- Tap val va test ---
# TODO: Dien phan cuoi test1 (tu split_idx tro di) de tao sequences val
X_val, y_val = create_sequences(???, y_test1[split_idx:], WINDOW_SIZE, STEP_EVAL)
X_test2_seq, y_test2_seq = create_sequences(X_test2_sc, y_test2, WINDOW_SIZE, step=STEP_EVAL)

print(f'Chuoi train : {X_tr.shape}  attack={(y_tr==1).sum()} ({100*(y_tr==1).mean():.1f}%)')
print(f'  -> Normal tu train_df: {len(X_norm_seq)}  |  Tu test1 70%%: {len(X_t1_tr)}')
print(f'Chuoi val   : {X_val.shape}  attack={(y_val==1).sum()} ({100*(y_val==1).mean():.1f}%)')
print(f'Chuoi test2 : {X_test2_seq.shape} attack={(y_test2_seq==1).sum()} ({100*(y_test2_seq==1).mean():.1f}%)')

## Phần 4: Kiến trúc 1D CNN

**Nguyên lý hoạt động:**
- **Conv1D(filters, kernel_size):** Bộ lọc trượt qua chuỗi, trích xuất đặc trưng cục bộ
- **BatchNormalization:** Chuẩn hóa activation, tăng tốc huấn luyện, giảm overfitting
- **MaxPooling1D:** Giảm chiều dài chuỗi (downsampling), giữ đặc trưng nổi bật nhất
- **GlobalAveragePooling1D:** Thay Flatten — tính trung bình toàn chuỗi, ít tham số hơn

**Kiến trúc mục tiêu:**
```
Input (30, 43) → Conv1D(64,3) → BN → MaxPool(2) → [14, 64]
             → Conv1D(128,3) → BN → MaxPool(2) → [7, 128]
             → Conv1D(64,3) → BN → GAP → [64]
             → Dense(64,relu) → Dropout(0.4) → Dense(1,sigmoid)
```

In [ ]:
def build_cnn_classifier(n_timesteps, n_features):
    model = Sequential([
        # Block 1: Conv1D đầu tiên với 64 filters
        Conv1D(filters=???, kernel_size=3, activation='relu',
               padding='same', input_shape=(n_timesteps, n_features)),
        BatchNormalization(),
        MaxPooling1D(pool_size=???),

        # Block 2: Conv1D thứ hai với 128 filters
        Conv1D(filters=???, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),

        # Block 3: Conv1D thứ ba với 64 filters
        Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),

        # Classification head
        GlobalAveragePooling1D(),
        Dense(???, activation='relu'),
        Dropout(???),
        Dense(1, activation='sigmoid')
    ], name='CNN_Classifier')

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model_cnn = build_cnn_classifier(WINDOW_SIZE, n_features)
model_cnn.summary()

## Phần 5: Huấn luyện Mô hình CNN

In [ ]:
pos_weight   = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
# TODO: Điền trọng số cho lớp tấn công
class_weight = {0: 1.0, 1: ???}
print(f'Class weight: normal=1.0, attack={pos_weight:.2f}')

callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True,
                  monitor='val_loss', verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=5,
                      min_lr=1e-6, verbose=0)
]

# TODO: Điền dữ liệu training vào fit
history = model_cnn.fit(
    ???, y_tr,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)
print(f'\nHuấn luyện kết thúc ở epoch {len(history.history["loss"])}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history['loss'],     label='Train Loss', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='crimson')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Curve — CNN'); axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='Train Acc', color='steelblue')
axes[1].plot(history.history['val_accuracy'], label='Val Acc',   color='crimson')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy Curve — CNN'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Phần 6: Tinh chỉnh Ngưỡng và Đánh giá

In [ ]:
y_prob_val = model_cnn.predict(X_val, verbose=0).ravel()
thresholds = np.arange(0.1, 0.9, 0.05)
# TODO: Điền ngưỡng t vào điều kiện so sánh
f1s        = [f1_score(y_val, (y_prob_val > ???).astype(int),
                       average='macro', zero_division=0) for t in thresholds]
# TODO: Điền danh sách f1s để tìm argmax
best_thr   = float(thresholds[np.argmax(???)])

plt.figure(figsize=(9, 4))
plt.plot(thresholds, f1s, 'o-', color='steelblue', linewidth=2)
plt.axvline(best_thr, color='crimson', linestyle='--', linewidth=2,
            label=f'Best threshold = {best_thr:.2f}')
plt.xlabel('Ngưỡng quyết định'); plt.ylabel('F1 (macro)')
plt.title('F1-score theo Ngưỡng quyết định (Validation) — CNN')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

y_pred_val = (y_prob_val > best_thr).astype(int)
f1  = f1_score(y_val, y_pred_val, average='macro', zero_division=0)
pre = precision_score(y_val, y_pred_val, pos_label=1, zero_division=0)
rec = recall_score(y_val, y_pred_val, pos_label=1, zero_division=0)
auc = roc_auc_score(y_val, y_prob_val)
print(f'\n=== CNN Classifier (Val, thr={best_thr:.2f}) ===')
print(f'  F1 (macro): {f1:.4f} | Precision: {pre:.4f} | Recall: {rec:.4f}')
print(f'  AUC-ROC  : {auc:.4f}')
cm = confusion_matrix(y_val, y_pred_val)
print(f'  TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')

## Phần 7: Đánh giá trên Test 2 (Held-out)

In [ ]:
# TODO: Điền tập test2 vào predict
y_prob_test2 = model_cnn.predict(???, verbose=0).ravel()
# TODO: Điền ngưỡng tốt nhất vào điều kiện so sánh
y_pred_test2 = (y_prob_test2 > ???).astype(int)

f1  = f1_score(y_test2_seq, y_pred_test2, average='macro', zero_division=0)
pre = precision_score(y_test2_seq, y_pred_test2, pos_label=1, zero_division=0)
rec = recall_score(y_test2_seq, y_pred_test2, pos_label=1, zero_division=0)
auc = roc_auc_score(y_test2_seq, y_prob_test2)
print(f'=== CNN Classifier (Test2, thr={best_thr:.2f}) ===')
print(f'  F1 (macro): {f1:.4f} | Precision: {pre:.4f} | Recall: {rec:.4f}')
print(f'  AUC-ROC  : {auc:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test2_seq, y_pred_test2)
ConfusionMatrixDisplay(cm, display_labels=['Bình thường', 'Tấn công']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix — CNN (Test 2, thr={best_thr:.2f})')

fpr, tpr, _ = roc_curve(y_test2_seq, y_prob_test2)
axes[1].plot(fpr, tpr, color='darkorange', linewidth=2.5,
             label=f'CNN (AUC={auc:.3f})')
axes[1].plot([0,1],[0,1],'k--', linewidth=1)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve — CNN (Test 2)'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Phần 8: So sánh và Phân tích

In [ ]:
# So sánh tổng hợp CNN vs các phương pháp trước
models_comparison = {
    'CNN (4)': {'F1': f1, 'Precision': pre, 'Recall': rec, 'AUC': auc}
}

print('=== SO SÁNH CÁC MÔ HÌNH (Test 2) ===')
comp_df = pd.DataFrame(models_comparison).T
print(comp_df.round(4).to_string())
print('\nGhi chú: Chạy B7 trước để so sánh đầy đủ với LSTM')

# Phân tích filter CNN (trọng số của Conv1D layer đầu tiên)
conv_weights = model_cnn.layers[0].get_weights()[0]  # shape: (kernel_size, n_features, n_filters)
print(f'\nTrọng số Conv1D đầu tiên: {conv_weights.shape}')
print('(kernel_size=3, n_features=43, n_filters=64)')

## Câu hỏi thảo luận

1. Tại sao `GlobalAveragePooling1D` được ưu tiên hơn `Flatten` trong CNN cho chuỗi thời gian?
2. `BatchNormalization` đặt trước hay sau activation? Ảnh hưởng như thế nào đến quá trình huấn luyện?
3. Tăng `kernel_size` lên 5 hoặc 7 sẽ ảnh hưởng đến khả năng phát hiện như thế nào?
4. Trong thực tế ICS, CNN hay LSTM phù hợp hơn cho real-time anomaly detection? Tại sao?